# Mid Fusion Lite - Kaggle Run All
Notebook nay duoc thiet ke de bam Run All la train duoc.

## Ban can gan 2 dataset trong Kaggle Notebook
1. Dataset code cua ban (chua folder `mid_fusion_lite`).
2. Dataset ISIC 2024 chua cac file metadata va HDF5 (`train-metadata.csv`, `test-metadata.csv`, `sample_submission.csv`, `train-image.hdf5`, `test-image.hdf5`).

## Quan trong
Chi can sua duong dan o Cell 2 (phan USER CONFIG).

In [ ]:
# =========================
# USER CONFIG (CHI SUA CELL NAY)
# =========================

# Preset: 'smoke' (nhanh de test), 'quick' (can bang), 'full_lite' (nang hon).
PRESET = 'smoke'

# Neu muon chi train 1 vai fold de test nhanh:
#   '0' hoac '0,1'
# Neu muon train tat ca fold cua preset: None
FOLDS = '0'

# Co the de None de dung mac dinh cua preset.
EPOCHS_OVERRIDE = None
IMAGE_SIZE_OVERRIDE = None

# Trong moi truong Kaggle khong internet, de False de tranh loi tai pretrained.
# Khi internet on va muon dung pretrained weights, set True.
USE_PRETRAINED_BACKBONE = False

# Neu ban co file weights backbone local trong Kaggle dataset, dien duong dan o day.
# Vi du: '/kaggle/input/my-backbone-weights/efficientnet_b0_backbone.pth'
BACKBONE_WEIGHTS_PATH = None

# ===== Duong dan can sua theo dataset cua ban tren Kaggle =====
# Vi du: /kaggle/input/my-mid-fusion-code/mid_fusion_lite
CODE_DATASET_ROOT = '/kaggle/input/your-code-dataset-name/mid_fusion_lite'

# Neu ban dung dataset competition goc: /kaggle/input/isic-2024-challenge
COMP_DATASET_ROOT = '/kaggle/input/isic-2024-challenge'

# Thu muc output tren working dir
OUTPUT_DIR = '/kaggle/working/mid_fusion_lite_outputs'

TRAIN_METADATA_PATH = f'{COMP_DATASET_ROOT}/train-metadata.csv'
TEST_METADATA_PATH = f'{COMP_DATASET_ROOT}/test-metadata.csv'
SAMPLE_SUBMISSION_PATH = f'{COMP_DATASET_ROOT}/sample_submission.csv'
TRAIN_HDF5_PATH = f'{COMP_DATASET_ROOT}/train-image.hdf5'
TEST_HDF5_PATH = f'{COMP_DATASET_ROOT}/test-image.hdf5'

In [ ]:
from pathlib import Path
import os

print('Kaggle input datasets:')
print(os.listdir('/kaggle/input'))

paths_to_check = {
    'CODE_DATASET_ROOT': CODE_DATASET_ROOT,
    'TRAIN_METADATA_PATH': TRAIN_METADATA_PATH,
    'TEST_METADATA_PATH': TEST_METADATA_PATH,
    'SAMPLE_SUBMISSION_PATH': SAMPLE_SUBMISSION_PATH,
    'TRAIN_HDF5_PATH': TRAIN_HDF5_PATH,
    'TEST_HDF5_PATH': TEST_HDF5_PATH,
}

missing = []
for key, value in paths_to_check.items():
    exists = Path(value).exists()
    print(f'{key}: {value} | exists={exists}')
    if not exists:
        missing.append(key)

if missing:
    raise FileNotFoundError(
        'Thieu duong dan: ' + ', '.join(missing) +
        '. Hay sua lai CELL 2 cho dung ten dataset/path.'
    )

print('Path check OK.')

In [ ]:
# Cai dependencies theo requirements trong folder code
!pip -q install -r "{CODE_DATASET_ROOT}/requirements.txt"

In [ ]:
import sys
from pathlib import Path

if CODE_DATASET_ROOT not in sys.path:
    sys.path.append(CODE_DATASET_ROOT)

from runner import run_with_preset

overrides = {
    'output_dir': OUTPUT_DIR,
    'train_metadata_path': TRAIN_METADATA_PATH,
    'test_metadata_path': TEST_METADATA_PATH,
    'sample_submission_path': SAMPLE_SUBMISSION_PATH,
    'train_hdf5_path': TRAIN_HDF5_PATH,
    'test_hdf5_path': TEST_HDF5_PATH,
    'use_pretrained_backbone': bool(USE_PRETRAINED_BACKBONE),
}

if BACKBONE_WEIGHTS_PATH is not None and str(BACKBONE_WEIGHTS_PATH).strip() != '':
    overrides['backbone_weights_path'] = str(BACKBONE_WEIGHTS_PATH)

if FOLDS is not None and str(FOLDS).strip() != '':
    overrides['fold_indices'] = [int(x.strip()) for x in str(FOLDS).split(',') if x.strip()]

if EPOCHS_OVERRIDE is not None:
    overrides['epochs'] = int(EPOCHS_OVERRIDE)

if IMAGE_SIZE_OVERRIDE is not None:
    overrides['image_size'] = int(IMAGE_SIZE_OVERRIDE)

print('Run config:')
print({'preset': PRESET, **overrides})

result = run_with_preset(PRESET, **overrides)

In [ ]:
import json
import pandas as pd
import shutil
from pathlib import Path

print('CV pAUC:', result['cv_pauc'])
print('OOF file:', result['oof_path'])
print('Submission file:', result['submission_path'])
print('Metrics file:', result['metrics_path'])

metrics = json.loads(Path(result['metrics_path']).read_text())
print('Fold metrics:')
for row in metrics.get('fold_metrics', []):
    print(row)

# Luu them mot file submission.csv o /kaggle/working de gui len competition nhanh
shutil.copy(result['submission_path'], '/kaggle/working/submission.csv')
print('Copied to /kaggle/working/submission.csv')

display(pd.read_csv('/kaggle/working/submission.csv').head())